# 🔄 Lab 1 : Fondations de l'Architecture Medallion

**Objectifs :**
- Créer son propre dataset (un seul par élève pour garder le projet propre)
- Ingérer des tables issues d'un dataset public dans sa couche Bronze (préfixe `bronze_`)
- Joindre ces tables avant transformation
- Nettoyer et filtrer des données brutes (Bronze ➔ Silver)
- Comprendre Vues vs Tables Matérialisées
- Agréger des données (Silver ➔ Gold)

## ⚠️ Règle d'or : Projet Partagé
Nous travaillons tous sur le même projet Google Cloud. **Pour ne pas polluer l'interface avec des dizaines de dossiers, vous allez créer UN SEUL dataset portant votre prénom.**

Vous utiliserez ensuite des **préfixes** sur vos tables pour identifier les couches Medallion (`bronze_`, `silver_`, `gold_`).

Exemple de dataset : `thelook_alice`
Exemple de tables : `thelook_alice.bronze_orders`, `thelook_alice.silver_clean_orders_items`.

### Étape 1 : Création de votre dataset
Vous devez créer votre dataset `thelook_prenom` dans la région `US`.
Google Cloud vous laisse le choix des armes. **Choisissez UNE des 3 méthodes ci-dessous** :

**Option A : Via la console BigQuery (Interface Graphique)**


**Option B : Via une requête SQL**

**Option C : Via le Terminal (Cloud Shell / CLI `bq mk`)**

### Étape 2 : Ingérer les données brutes (Couche Bronze)
Dans une vraie architecture Medallion, la couche Bronze est une copie **exacte** (1:1) des systèmes sources. Nous allons utiliser le dataset public e-commerce de Google : `bigquery-public-data.thelook_ecommerce`.

L'objectif ici est d'**ingérer (copier)** les tables `orders` et `order_items` dans VOTRE dataset, en les préfixant par `bronze_`.

### ✍️ À vous de jouer (Exercice 1 : Ingestion Bronze)
**Consignes :**
1. Copiez la table `orders` vers `thelook_prenom.bronze_orders`.
2. Copiez la table `order_items` vers `thelook_prenom.bronze_order_items`.

### Étape 3 : Joindre les tables sources
Maintenant que les tables sont chez vous, regroupons-les en une seule table brute unifiée avant de commencer le nettoyage.

### ✍️ À vous de jouer (Exercice 2 : Jointure)
**Consignes :**
1. Créez une table `thelook_prenom.bronze_raw_orders_items`.
2. Faites une jointure (`JOIN`) entre vos tables locales `thelook_prenom.bronze_orders` (alias `o`) et `thelook_prenom.bronze_order_items` (alias `oi`) sur la clé `order_id`.
3. Sélectionnez AU MINIMUM ces colonnes : `o.order_id`, `o.user_id`, `o.created_at`, `o.status`, `oi.id AS order_item_id`, `oi.product_id`, `oi.sale_price`. Vous pouvez décider d'ajouter d'autres colonnes pertinentes.

## Partie 1 : Transformation Bronze ➔ Silver
Dans la couche Silver, on nettoie, on type correctement les données et on filtre ce qui n'est pas pertinent.

**Exemple de nettoyage (CAST, COALESCE) :**
```sql
SELECT 
  order_id,
  DATE(created_at) AS date_propre,
  COALESCE(sale_price, 0.0) AS prix_propre
FROM `thelook_prenom.bronze_raw_orders_items`
WHERE status IS NOT NULL;
```

### ✍️ À vous de jouer (Exercice 3 : Nettoyage Silver)
**Consignes :** 
_vous allez renommer chaque colonne en français_
1. Créez une table `thelook_prenom.silver_clean_orders_items` à partir de `thelook_prenom.bronze_raw_orders_items`.
2. Convertissez le timestamp `created_at` en date simple. (voir documentation pour les functions relatives aux dates)
3. Gardez uniquement les commandes dont le statut (`status`) est **'Complete'** ou **'Shipped'**.
4. Excluez les articles dont le prix (`sale_price`) est nul ou négatif.

In [ ]:
-- Écrivez votre requête de transformation Silver ici :
CREATE OR REPLACE TABLE `thelook_prenom.silver_clean_orders_items` AS
SELECT
  ...


## Partie 2 : Vue ou Table Matérialisée ?

Avant de créer notre vue, faisons un court rappel théorique sur les différences entre une **Vue standard** et une **Vue matérialisée** dans BigQuery :

| Caractéristique | Vue standard (`CREATE VIEW`) | Vue matérialisée (`CREATE MATERIALIZED VIEW`) |
| :--- | :--- | :--- |
| **Stockage** | Aucun (c'est juste une requête sauvegardée) | Stocke physiquement les résultats pré-calculés |
| **Coût de facturation** | Scan total des tables sous-jacentes à chaque fois | Réduit (lit les données pré-calculées + le delta récent) |
| **Performance** | Plus lent (recalcule tout à la volée) | Très rapide (zéro-maintenance, cache intelligent) |
| **Mise à jour** | Toujours 100% en temps réel | Automatique en arrière-plan (BigQuery gère le rafraîchissement) |
| **Cas d'usage idéal** | Logique métier de base (Silver), restriction d'accès | Tableaux de bord (Gold), agrégations coûteuses appelées souvent |

---
Au lieu d'une table physique qui doit être recalculée manuellement, on peut créer une vue matérialisée pour la couche Silver, ou une vue simple pour des requêtes de préparation.

**Exemple de Vue Matérialisée :**
```sql
CREATE MATERIALIZED VIEW `thelook_prenom.silver_mv_shipped_orders` AS
SELECT order_id, user_id FROM `thelook_prenom.bronze_raw_orders_items` WHERE status = 'Shipped';
```

### ✍️ À vous de jouer (Exercice 4 : Création d'une Vue)
**Consignes :**
Créez une vue classique (`CREATE OR REPLACE VIEW`) nommée `thelook_prenom.silver_v_high_value_items` qui ne sélectionne que les articles de `silver_clean_orders_items` dont le `sale_price` est supérieur à 100$. Interrogez ensuite votre vue (via un `SELECT *`).

In [ ]:
-- Écrivez votre création de vue ici :



## Partie 3 : Agrégation (Silver ➔ Gold)
La couche Gold contient les indicateurs métier, souvent hautement agrégés pour les tableaux de bord (BI).

**Exemple d'agrégation :**
```sql
SELECT user_id, COUNT(order_item_id) AS nb_items_bought
FROM `thelook_prenom.silver_clean_orders_items`
GROUP BY user_id;
```

### ✍️ À vous de jouer (Exercice 5 : Création du Mart Gold)
**Consignes :**
Créez une table `thelook_prenom.gold_daily_revenue` qui calcule :
- Le chiffre d'affaires total (`SUM(sale_price)` nommé `total_revenue`)
- Le nombre d'articles uniques vendus (`COUNT(order_item_id)` nommé `items_sold`)
Tout cela **par jour** (`order_date`). Triez le résultat du plus récent au plus ancien (`ORDER BY order_date DESC`).

In [ ]:
-- Écrivez votre requête d'agrégation Gold ici :

